# Three leading alternatives on the fixed training-only benchmark

Evaluate GMM tied k10, K-means k8, and normalized DBSCAN/logistic C100 on exactly the same saved 40 splits as training-only Ani and original DBSCAN. All candidate settings are frozen; held-out and Kaggle test inputs never enter fitting. No submission is generated.

## Fixed candidates and inherited training-only protocol

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import f1_score
from sklearn.preprocessing import Normalizer, StandardScaler
from threadpoolctl import threadpool_limits

REASONS=['Safety','Funding','Efficacy','Recruitment','Other']
FEATURES=[f'emb_{i}' for i in range(150)]
NEW_RECIPES=['gmm_tied_k10','kmeans_k8','dbscan_normalized_C100']
REFERENCES=['ani_train_only','dbscan_train_only']
cwd=Path.cwd()
DATA=next(p for p in [cwd,cwd/'part1',cwd/'AML-Midterm'/'part1']
          if (p/'experiments/paired_recipe_cv/active_benchmark.json').is_file()).resolve()
BASE=DATA/'experiments/paired_recipe_cv'
OUT=DATA/'experiments/leading_recipes_cv'
for folder in ['models','clustering','datasets','predictions','controls']:
    (OUT/folder).mkdir(parents=True,exist_ok=True)
active=json.loads((BASE/'active_benchmark.json').read_text())
assert active['active_comparison']=='shared_train_only'
assert not active['held_features_used_for_fit'] and not active['test_features_used_for_fit']
base_settings=json.loads((BASE/'settings.json').read_text())
raw_path=BASE/'datasets/shared_raw_features_labels.csv'
assert hashlib.sha256(raw_path.read_bytes()).hexdigest()==base_settings['dataset_sha256']
raw=pd.read_csv(raw_path,float_precision='round_trip')
X=raw[FEATURES].to_numpy();Y=raw[REASONS].to_numpy(dtype=int)
known=np.isin(Y,[0,1]).all(axis=1)
original_known=np.flatnonzero(known & (raw.source_group.to_numpy()=='original_training'))
unlabeled=np.flatnonzero((~known)&(raw.source_group.to_numpy()=='original_training'))
validation=np.flatnonzero(raw.source_group.to_numpy()=='former_validation')
test=np.flatnonzero(raw.source_group.to_numpy()=='test')
assert len(original_known)==157 and len(validation)==78 and len(unlabeled)==1014 and len(test)==312
assert known.sum()==235 and (Y[~known]==-100).all()
METHOD=DATA/'experiments/method_comparison'
cluster_paths=dict(gmm_tied_k10=METHOD/'clustering_models/gmm_tied_k10.joblib',
                   kmeans_k8=METHOD/'clustering_models/kmeans_k08.joblib')
cluster_prototypes={k:joblib.load(v) for k,v in cluster_paths.items()}
source_paths=dict(gmm_tied_k10=METHOD/'classifier_models/gmm_tied_k10__logistic_C1000.joblib',
                  kmeans_k8=METHOD/'classifier_models/kmeans_k08__logistic_C1000.joblib',
                  dbscan_normalized_C100=DATA/'experiments/classifier_cross_validation/models/full_training/normalized_C100.joblib')
prototypes={k:joblib.load(v) for k,v in source_paths.items()}
source_datasets=dict(gmm_tied_k10=METHOD/'datasets/gmm_tied_k10.csv',
                     kmeans_k8=METHOD/'datasets/kmeans_k08.csv',
                     dbscan_normalized_C100=DATA/'experiments/additional_methods/datasets/dbscan_m5_q25.csv')
RULE=dict(minimum_support=3.,prior_strength=2.,positive_threshold=.6,negative_threshold=.1,
          maximum_generated_weight=.25,maximum_class_fraction=.5)
assert all(p['label_rule']==RULE for p in cluster_prototypes.values())
assert cluster_prototypes['gmm_tied_k10']['clustering'].get_params()['covariance_type']=='tied'
assert cluster_prototypes['kmeans_k8']['clustering'].n_clusters==8
assert prototypes['dbscan_normalized_C100']['estimators'][0].C==100.
protected_paths=[raw_path,BASE/'active_benchmark.json',BASE/'comparison.csv',BASE/'fold_scores.csv']
protected_paths += list(source_paths.values())+list(cluster_paths.values())+list(source_datasets.values())
protected_paths += sorted((BASE/'splits').glob('repeat?.csv'))
PROTECTED={str(p.relative_to(DATA)):hashlib.sha256(p.read_bytes()).hexdigest() for p in protected_paths}
PROTOCOL=dict(
    active_policy='training-only',new_recipes=NEW_RECIPES,references=REFERENCES,label_pool=235,
    folds=5,repeats=8,seeds=list(range(100,108)),saved_splits=str((BASE/'splits').relative_to(DATA)),
    verified_fit_rows=188,held_rows=47,unlabeled_training_rows=1014,clustering_fit_rows=1202,
    held_features_used_for_fit=False,test_features_used_for_fit=False,
    recipe_selection='GMM tied k10 and K-means k8 led prior distinct clustering alternatives; normalized DBSCAN C100 was the previous CV-selected normalization contender. Choices frozen before these scores.',
    gmm_parameters=cluster_prototypes['gmm_tied_k10']['clustering'].get_params(),
    kmeans_parameters=cluster_prototypes['kmeans_k8']['clustering'].get_params(),
    classifier_C=dict(gmm_tied_k10=1000.,kmeans_k8=1000.,dbscan_normalized_C100=100.),
    classifier_solver='lbfgs',classifier_tol=1e-4,classifier_max_iter=5000,label_rule=RULE,
    normalized_recipe='Reuse fold-specific DBSCAN targets/raw weights and verified-only StandardScaler, then row L2 Normalizer; refit five logistic classifiers at C100',
    pseudo_label_rule='Rebuild GMM/K-means label mapping using only 188 fitting labels; cap raw generated weights at .25 and preserve known class balance; per-reason weights normalize to188',
    cutoff=.5,highest_probability_fallback=True,
    reference_models='Reuse exact saved training-only Ani and DBSCAN held-out predictions on the identical splits',
    primary_metric='Pool235 out-of-fold predictions within each repeat, then mean sample F1 over8 repeats',
    no_tuning=True,no_new_submission=True,no_kaggle_upload=True,protected_input_sha256=PROTECTED)
(OUT/'protocol.json').write_text(json.dumps(PROTOCOL,indent=2)+'\n')
(OUT/'PROTOCOL.md').write_text('# Three leading alternatives on the fixed training-only benchmark\n\n'
    'Freeze GMM tied k=10 + logistic C1000, K-means k=8 + logistic C1000, and normalized DBSCAN + logistic C100. '
    'No parameter search. Reuse the exact 40 saved folds (235 known labels,188 fit/47 held out; eight repeats of five folds). '
    'All feature learning and label generation excludes the held-out trials and all Kaggle test trials. '
    'Clustering uses the 188 known + 1,014 originally unlabeled training trials. '
    'Score each recipe using pooled trial sample F1 per repeat, then the eight-repeat mean, with paired fold/repeat differences '
    'against the frozen training-only Ani and original DBSCAN references. Old single-validation scores are reproduction controls only.\n')
print('Frozen before scoring:',NEW_RECIPES)
print('Exactly the existing 40 training-only folds; no held-out/test features used for fitting.')



Frozen before scoring: ['gmm_tied_k10', 'kmeans_k8', 'dbscan_normalized_C100']
Exactly the existing 40 training-only folds; no held-out/test features used for fitting.


## Label generation, classifier fitting, and scoring helpers

In [2]:
def decide(probability,cutoff=.5):
    assert np.isfinite(probability).all() and ((probability>=0)&(probability<=1)).all()
    prediction=(probability>=cutoff).astype(int);empty=prediction.sum(axis=1)==0
    ix=np.flatnonzero(empty);prediction[ix,probability[ix].argmax(axis=1)]=1
    return prediction,empty


def score(actual,probability,cutoff=.5):
    predicted,empty=decide(probability,cutoff)
    return dict(sample_f1=float(f1_score(actual,predicted,average='samples',zero_division=0)),
                macro_f1=float(f1_score(actual,predicted,average='macro',zero_division=0)),
                fallback_trials=int(empty.sum()),
                **dict(zip([f'f1_{r}' for r in REASONS],f1_score(actual,predicted,average=None,zero_division=0).tolist())))


def predict(packet,features):
    transformed=packet['scaler'].transform(features)
    if packet.get('normalizer') is not None:transformed=packet['normalizer'].transform(transformed)
    return np.column_stack([e.predict_proba(transformed)[:,list(e.classes_).index(1)] for e in packet['estimators']])


def generate_targets(responsibility,pool,fitting):
    verified=np.isin(pool,fitting)
    truth=Y[pool[verified]]
    support_component=responsibility[verified].sum(axis=0)
    positives=responsibility[verified].T@truth
    rates=(positives+2*truth.mean(axis=0))/(support_component[:,None]+2)
    probability=responsibility@rates;support=responsibility@support_component
    targets=np.full((len(pool),5),-100,dtype=int);weights=np.zeros((len(pool),5))
    targets[verified]=truth;weights[verified]=1.
    eligible=(~verified)&(support>=3)
    for j in range(5):
        pos=eligible&(probability[:,j]>=.6);neg=eligible&(probability[:,j]<=.1)
        npos,nneg=int(pos.sum()),int(neg.sum());kp=int(truth[:,j].sum());kn=int(len(truth)-kp)
        if not npos or not nneg:continue
        fraction=min(.5,.25*npos/kp,.25*nneg/kn)
        targets[pos,j]=1;targets[neg,j]=0;weights[pos,j]=fraction*kp/npos;weights[neg,j]=fraction*kn/nneg
        assert weights[~verified,j].max()<=.25+1e-12
        np.testing.assert_allclose(weights[pos,j].sum()/kp,weights[neg,j].sum()/kn,atol=1e-12)
    np.testing.assert_array_equal(targets[verified],truth)
    np.testing.assert_array_equal(weights[verified],np.ones((len(fitting),5)))
    assert np.array_equal(weights>0,targets!=-100)
    return targets,weights,probability,support,rates,support_component


FIT_AUDIT=[];LABEL_AUDIT=[];FOLD_ROWS=[];ROLE_ROWS=[]

def fit_classifier(recipe,context,pool,fitting,targets,weights,scaler,persist=True):
    assert not len(np.intersect1d(pool,test))
    np.testing.assert_array_equal(targets[np.isin(pool,fitting)],Y[pool[np.isin(pool,fitting)]])
    assert np.array_equal(targets!=-100,weights>0)
    normalized_weights=weights*len(fitting)/weights.sum(axis=0)
    np.testing.assert_allclose(normalized_weights.sum(axis=0),len(fitting),atol=1e-10)
    design=scaler.transform(X[pool]);normalizer=None
    if recipe=='dbscan_normalized_C100':
        normalizer=Normalizer(norm='l2');design=normalizer.transform(design)
        np.testing.assert_allclose(np.linalg.norm(design,axis=1),1.,atol=1e-12)
    estimators=[]
    with threadpool_limits(limits=1):
        for j,reason in enumerate(REASONS):
            included=weights[:,j]>0;estimator=clone(prototypes[recipe]['estimators'][j])
            with warnings.catch_warnings():
                warnings.simplefilter('error',ConvergenceWarning)
                estimator.fit(design[included],targets[included,j],sample_weight=normalized_weights[included,j])
            estimators.append(estimator)
            if persist:
                FIT_AUDIT.append(dict(context=context,recipe=recipe,reason=reason,
                                      fitting_rows=int(included.sum()),verified_rows=len(fitting),
                                      generated_rows=int((included&~np.isin(pool,fitting)).sum()),
                                      C=float(estimator.C),tol=float(estimator.tol),iterations=int(estimator.n_iter_.max()),
                                      weight_total=float(normalized_weights[included,j].sum())))
    packet=dict(kind='logistic',recipe=recipe,scaler=scaler,normalizer=normalizer,estimators=estimators,
                features=FEATURES,reasons=REASONS,fitting_rows=fitting,pool_rows=pool,cutoff=.5,nonempty_fallback=True)
    frame=raw.iloc[pool][['ID']].copy();frame['global_row']=pool;frame['verified_fit']=np.isin(pool,fitting)
    for j,reason in enumerate(REASONS):
        frame[reason]=targets[:,j];frame[f'weight_{reason}']=weights[:,j]
        frame[f'normalized_weight_{reason}']=normalized_weights[:,j]
        if persist:
            LABEL_AUDIT.append(dict(context=context,recipe=recipe,reason=reason,
                                    generated_positive=int(((targets[:,j]==1)&~frame.verified_fit.to_numpy()).sum()),
                                    generated_negative=int(((targets[:,j]==0)&~frame.verified_fit.to_numpy()).sum())))
    if persist:frame.to_csv(OUT/'datasets'/f'{context}__{recipe}.csv',index=False)
    return packet,frame


def fit_cluster_recipe(recipe,context,pool,fitting,classifier_scaler,cluster_scaler,persist=True):
    assert int(cluster_scaler.n_samples_seen_)==len(pool)
    clustering=clone(cluster_prototypes[recipe]['clustering'])
    transformed=cluster_scaler.transform(X[pool])
    with threadpool_limits(limits=1),warnings.catch_warnings():
        warnings.simplefilter('error',ConvergenceWarning)
        clustering.fit(transformed)
        if recipe=='gmm_tied_k10':
            assert clustering.converged_
            responsibility=clustering.predict_proba(transformed)
        else:
            responsibility=np.eye(clustering.n_clusters)[clustering.labels_]
    targets,weights,probability,support,rates,component_support=generate_targets(responsibility,pool,fitting)
    packet,frame=fit_classifier(recipe,context,pool,fitting,targets,weights,classifier_scaler,persist)
    if persist:
        joblib.dump(dict(clustering=clustering,scaler=cluster_scaler,pool_rows=pool,fitting_rows=fitting,
                         label_rule=RULE),OUT/'clustering'/f'{context}__{recipe}.joblib',compress=3)
        audit=raw.iloc[pool][['ID']].copy();audit['effective_labeled_support']=support
        for k in range(responsibility.shape[1]):audit[f'membership_{k}']=responsibility[:,k]
        for j,reason in enumerate(REASONS):audit[f'estimated_probability_{reason}']=probability[:,j]
        audit.to_csv(OUT/'clustering'/f'{context}__{recipe}_memberships.csv',index=False)
        components=pd.DataFrame(dict(component=np.arange(len(component_support)),verified_support=component_support))
        for j,reason in enumerate(REASONS):components[f'rate_{reason}']=rates[:,j]
        components.to_csv(OUT/'clustering'/f'{context}__{recipe}_rates.csv',index=False)
    return packet,frame


def persist_score(recipe,context,repeat,fold,packet,held):
    path=OUT/'models'/f'{context}__{recipe}.joblib';joblib.dump(packet,path,compress=3)
    probability=predict(packet,X[held])
    np.testing.assert_allclose(predict(joblib.load(path),X[held]),probability,rtol=1e-12,atol=1e-12)
    predicted,empty=decide(probability)
    frame=raw.iloc[held][['ID','source_group']].copy();frame['global_row']=held
    for j,reason in enumerate(REASONS):
        frame[f'actual_{reason}']=Y[held,j];frame[f'probability_{reason}']=probability[:,j]
        frame[f'predicted_{reason}']=predicted[:,j]
    frame['used_fallback']=empty
    frame['trial_f1']=2*(predicted*Y[held]).sum(axis=1)/(predicted.sum(axis=1)+Y[held].sum(axis=1))
    frame.to_csv(OUT/'predictions'/f'{context}__{recipe}.csv',index=False)
    FOLD_ROWS.append(dict(context=context,repeat=repeat,fold=fold,recipe=recipe,**score(Y[held],probability)))



## Reproduce the three saved validation-stage models

In [3]:
# Reproduce all three existing 157-label models before scoring any new folds.
pool=np.sort(np.r_[original_known,unlabeled])
classifier_scaler=StandardScaler().fit(X[original_known]);cluster_scaler=StandardScaler().fit(X[pool])
controls=[]
for recipe in NEW_RECIPES:
    if recipe=='dbscan_normalized_C100':
        fixed=pd.read_csv(source_datasets[recipe],float_precision='round_trip')
        assert fixed.ID.tolist()==raw.iloc[pool].ID.tolist()
        packet,frame=fit_classifier(recipe,'validation_control',pool,original_known,
            fixed[REASONS].to_numpy(dtype=int),fixed[[f'weight_{r}' for r in REASONS]].to_numpy(),classifier_scaler,False)
    else:
        packet,frame=fit_cluster_recipe(recipe,'validation_control',pool,original_known,classifier_scaler,cluster_scaler,False)
        fixed=pd.read_csv(source_datasets[recipe],float_precision='round_trip')
        assert fixed.ID.tolist()==frame.ID.tolist()
        np.testing.assert_array_equal(frame[REASONS],fixed[REASONS])
        np.testing.assert_allclose(frame[[f'weight_{r}' for r in REASONS]],fixed[[f'weight_{r}' for r in REASONS]],atol=1e-12)
    probability=predict(packet,X[validation]);original_probability=predict(prototypes[recipe],X[validation])
    np.testing.assert_allclose(probability,original_probability,rtol=1e-8,atol=1e-10)
    controls.append(dict(recipe=recipe,**score(Y[validation],probability),probability_reproduced=True))
    frame.to_csv(OUT/'controls'/f'{recipe}_targets_weights.csv',index=False)
    joblib.dump(packet,OUT/'controls'/f'{recipe}_classifier.joblib',compress=3)
pd.DataFrame(controls).to_csv(OUT/'controls/reproduction.csv',index=False)
display(pd.DataFrame(controls))
print('All three validation-stage sources reproduced; no settings selected from these control scores.')



,recipe,sample_f1,macro_f1,fallback_trials,f1_Safety,f1_Funding,f1_Efficacy,f1_Recruitment,f1_Other,probability_reproduced
0,gmm_tied_k10,0.905983,0.811626,4,0.941176,0.666667,0.838710,0.980000,0.631579,True
1,kmeans_k8,0.901709,0.809077,5,0.941176,0.666667,0.866667,0.970874,0.600000,True
2,dbscan_normalized_C100,0.829060,0.584852,13,0.875000,0.000000,0.666667,0.961538,0.421053,True


All three validation-stage sources reproduced; no settings selected from these control scores.


## Run each repeat on the inherited folds

In [4]:
def run_repeat(repeat):
    assignment=pd.read_csv(BASE/'splits'/f'repeat{repeat}.csv')
    for fold in range(5):
        context=f'repeat{repeat}_fold{fold}'
        held=raw.index[raw.ID.isin(assignment.loc[assignment.fold==fold,'ID'])].to_numpy()
        fitting=raw.index[raw.ID.isin(assignment.loc[assignment.fold!=fold,'ID'])].to_numpy()
        assert len(held)==47 and len(fitting)==188
        pool=np.sort(np.r_[fitting,unlabeled]);assert len(pool)==1202
        assert not len(np.intersect1d(pool,np.r_[held,test]))
        prep=joblib.load(BASE/'preprocessing'/f'{context}_train_only.joblib')
        np.testing.assert_array_equal(prep['feature_rows'],pool)
        # Refit the verified-only classifier scaler. Verify the cached cluster
        # scaler is exactly the fit for this same fold before reusing it.
        classifier_scaler=StandardScaler().fit(X[fitting])
        recomputed_cluster_scaler=StandardScaler().fit(X[pool])
        np.testing.assert_allclose(recomputed_cluster_scaler.mean_,prep['scaler'].mean_,atol=1e-14)
        np.testing.assert_allclose(recomputed_cluster_scaler.scale_,prep['scaler'].scale_,atol=1e-14)
        cluster_scaler=recomputed_cluster_scaler
        dbscan_source=pd.read_csv(BASE/'datasets'/f'{context}__dbscan_train_only.csv',float_precision='round_trip')
        assert dbscan_source.ID.tolist()==raw.iloc[pool].ID.tolist()
        np.testing.assert_array_equal(dbscan_source.verified_fit.to_numpy(),np.isin(pool,fitting))
        for recipe in NEW_RECIPES:
            if recipe=='dbscan_normalized_C100':
                packet,frame=fit_classifier(recipe,context,pool,fitting,
                    dbscan_source[REASONS].to_numpy(dtype=int),
                    dbscan_source[[f'weight_{r}' for r in REASONS]].to_numpy(),classifier_scaler)
            else:
                packet,frame=fit_cluster_recipe(recipe,context,pool,fitting,classifier_scaler,cluster_scaler)
            assert set(frame.ID).isdisjoint(raw.iloc[held].ID)
            assert set(frame.ID).isdisjoint(raw.iloc[test].ID)
            persist_score(recipe,context,repeat,fold,packet,held)
            ROLE_ROWS.append(dict(context=context,recipe=recipe,verified_fit_rows=188,held_rows=47,
                                  pool_rows=1202,held_fit_overlap=0,test_fit_overlap=0))
        for reference in REFERENCES:
            frame=pd.read_csv(BASE/'predictions'/f'{context}__{reference}.csv',float_precision='round_trip')
            assert set(frame.ID)==set(raw.iloc[held].ID)
            frame.to_csv(OUT/'predictions'/f'{context}__{reference}.csv',index=False)
            actual=frame[[f'actual_{r}' for r in REASONS]].to_numpy(dtype=int)
            probability=frame[[f'probability_{r}' for r in REASONS]].to_numpy()
            FOLD_ROWS.append(dict(context=context,repeat=repeat,fold=fold,recipe=reference,
                                  **score(actual,probability,.3 if reference=='ani_train_only' else .5)))
        pd.DataFrame(FOLD_ROWS).to_csv(OUT/'fold_scores.csv',index=False)
        pd.DataFrame(FIT_AUDIT).to_csv(OUT/'fit_audit.csv',index=False)
        pd.DataFrame(LABEL_AUDIT).to_csv(OUT/'generated_labels.csv',index=False)
        pd.DataFrame(ROLE_ROWS).to_csv(OUT/'role_audit.csv',index=False)
        print(context,{row['recipe']:round(row['sample_f1'],5) for row in FOLD_ROWS[-5:]},flush=True)



## Repeat 1 of 8: three candidates and two frozen references

In [5]:
run_repeat(0)




repeat0_fold0 {'gmm_tied_k10': 0.85106, 'kmeans_k8': 0.87943, 'dbscan_normalized_C100': 0.83688, 'ani_train_only': 0.85106, 'dbscan_train_only': 0.85106}


repeat0_fold1 {'gmm_tied_k10': 0.91489, 'kmeans_k8': 0.9078, 'dbscan_normalized_C100': 0.87234, 'ani_train_only': 0.90071, 'dbscan_train_only': 0.88652}


repeat0_fold2 {'gmm_tied_k10': 0.89362, 'kmeans_k8': 0.89362, 'dbscan_normalized_C100': 0.92199, 'ani_train_only': 0.95035, 'dbscan_train_only': 0.89362}


repeat0_fold3 {'gmm_tied_k10': 0.90071, 'kmeans_k8': 0.9078, 'dbscan_normalized_C100': 0.92908, 'ani_train_only': 0.92199, 'dbscan_train_only': 0.9078}


repeat0_fold4 {'gmm_tied_k10': 0.92908, 'kmeans_k8': 0.92908, 'dbscan_normalized_C100': 0.95745, 'ani_train_only': 0.95745, 'dbscan_train_only': 0.92908}


## Repeat 2 of 8: three candidates and two frozen references

In [6]:
run_repeat(1)




repeat1_fold0 {'gmm_tied_k10': 0.87943, 'kmeans_k8': 0.87943, 'dbscan_normalized_C100': 0.85106, 'ani_train_only': 0.93617, 'dbscan_train_only': 0.87943}


repeat1_fold1 {'gmm_tied_k10': 0.82624, 'kmeans_k8': 0.86525, 'dbscan_normalized_C100': 0.89362, 'ani_train_only': 0.91489, 'dbscan_train_only': 0.85816}


repeat1_fold2 {'gmm_tied_k10': 0.90355, 'kmeans_k8': 0.91773, 'dbscan_normalized_C100': 0.93617, 'ani_train_only': 0.92199, 'dbscan_train_only': 0.88936}


repeat1_fold3 {'gmm_tied_k10': 0.94326, 'kmeans_k8': 0.94326, 'dbscan_normalized_C100': 0.88652, 'ani_train_only': 0.95035, 'dbscan_train_only': 0.92199}


repeat1_fold4 {'gmm_tied_k10': 0.85816, 'kmeans_k8': 0.88652, 'dbscan_normalized_C100': 0.86525, 'ani_train_only': 0.86525, 'dbscan_train_only': 0.90071}


## Repeat 3 of 8: three candidates and two frozen references

In [7]:
run_repeat(2)




repeat2_fold0 {'gmm_tied_k10': 0.9078, 'kmeans_k8': 0.93617, 'dbscan_normalized_C100': 0.87943, 'ani_train_only': 0.87943, 'dbscan_train_only': 0.93617}


repeat2_fold1 {'gmm_tied_k10': 0.8227, 'kmeans_k8': 0.8617, 'dbscan_normalized_C100': 0.88652, 'ani_train_only': 0.91489, 'dbscan_train_only': 0.84752}


repeat2_fold2 {'gmm_tied_k10': 0.82624, 'kmeans_k8': 0.84043, 'dbscan_normalized_C100': 0.82979, 'ani_train_only': 0.85461, 'dbscan_train_only': 0.8227}


repeat2_fold3 {'gmm_tied_k10': 0.85461, 'kmeans_k8': 0.89362, 'dbscan_normalized_C100': 0.95745, 'ani_train_only': 0.95745, 'dbscan_train_only': 0.91773}


repeat2_fold4 {'gmm_tied_k10': 0.88652, 'kmeans_k8': 0.89362, 'dbscan_normalized_C100': 0.87234, 'ani_train_only': 0.88652, 'dbscan_train_only': 0.89362}


## Repeat 4 of 8: three candidates and two frozen references

In [8]:
run_repeat(3)




repeat3_fold0 {'gmm_tied_k10': 0.90426, 'kmeans_k8': 0.90426, 'dbscan_normalized_C100': 0.85106, 'ani_train_only': 0.89007, 'dbscan_train_only': 0.88298}


repeat3_fold1 {'gmm_tied_k10': 0.92908, 'kmeans_k8': 0.92908, 'dbscan_normalized_C100': 0.96454, 'ani_train_only': 0.92908, 'dbscan_train_only': 0.91489}


repeat3_fold2 {'gmm_tied_k10': 0.88652, 'kmeans_k8': 0.85816, 'dbscan_normalized_C100': 0.84397, 'ani_train_only': 0.84397, 'dbscan_train_only': 0.86525}


repeat3_fold3 {'gmm_tied_k10': 0.92199, 'kmeans_k8': 0.89007, 'dbscan_normalized_C100': 0.85106, 'ani_train_only': 0.9078, 'dbscan_train_only': 0.87589}


repeat3_fold4 {'gmm_tied_k10': 0.90426, 'kmeans_k8': 0.90426, 'dbscan_normalized_C100': 0.9078, 'ani_train_only': 0.93617, 'dbscan_train_only': 0.89716}


## Repeat 5 of 8: three candidates and two frozen references

In [9]:
run_repeat(4)




repeat4_fold0 {'gmm_tied_k10': 0.90071, 'kmeans_k8': 0.92908, 'dbscan_normalized_C100': 0.92908, 'ani_train_only': 0.9539, 'dbscan_train_only': 0.9078}


repeat4_fold1 {'gmm_tied_k10': 0.85816, 'kmeans_k8': 0.9078, 'dbscan_normalized_C100': 0.91489, 'ani_train_only': 0.87943, 'dbscan_train_only': 0.9078}


repeat4_fold2 {'gmm_tied_k10': 0.85106, 'kmeans_k8': 0.85106, 'dbscan_normalized_C100': 0.84397, 'ani_train_only': 0.90071, 'dbscan_train_only': 0.85106}


repeat4_fold3 {'gmm_tied_k10': 0.9461, 'kmeans_k8': 0.92482, 'dbscan_normalized_C100': 0.9461, 'ani_train_only': 0.91489, 'dbscan_train_only': 0.89645}


repeat4_fold4 {'gmm_tied_k10': 0.91489, 'kmeans_k8': 0.9078, 'dbscan_normalized_C100': 0.86525, 'ani_train_only': 0.91135, 'dbscan_train_only': 0.90071}


## Repeat 6 of 8: three candidates and two frozen references

In [10]:
run_repeat(5)




repeat5_fold0 {'gmm_tied_k10': 0.90071, 'kmeans_k8': 0.89362, 'dbscan_normalized_C100': 0.91489, 'ani_train_only': 0.88652, 'dbscan_train_only': 0.88652}


repeat5_fold1 {'gmm_tied_k10': 0.85461, 'kmeans_k8': 0.88652, 'dbscan_normalized_C100': 0.82979, 'ani_train_only': 0.88652, 'dbscan_train_only': 0.85106}


repeat5_fold2 {'gmm_tied_k10': 0.86525, 'kmeans_k8': 0.9078, 'dbscan_normalized_C100': 0.92199, 'ani_train_only': 0.91489, 'dbscan_train_only': 0.88652}


repeat5_fold3 {'gmm_tied_k10': 0.9078, 'kmeans_k8': 0.95035, 'dbscan_normalized_C100': 0.92908, 'ani_train_only': 0.95035, 'dbscan_train_only': 0.93617}


repeat5_fold4 {'gmm_tied_k10': 0.92199, 'kmeans_k8': 0.89362, 'dbscan_normalized_C100': 0.90071, 'ani_train_only': 0.90071, 'dbscan_train_only': 0.88652}


## Repeat 7 of 8: three candidates and two frozen references

In [11]:
run_repeat(6)




repeat6_fold0 {'gmm_tied_k10': 0.88652, 'kmeans_k8': 0.86525, 'dbscan_normalized_C100': 0.88652, 'ani_train_only': 0.90071, 'dbscan_train_only': 0.88652}


repeat6_fold1 {'gmm_tied_k10': 0.91489, 'kmeans_k8': 0.90071, 'dbscan_normalized_C100': 0.93617, 'ani_train_only': 0.92199, 'dbscan_train_only': 0.88227}


repeat6_fold2 {'gmm_tied_k10': 0.83688, 'kmeans_k8': 0.83688, 'dbscan_normalized_C100': 0.89362, 'ani_train_only': 0.91489, 'dbscan_train_only': 0.83688}


repeat6_fold3 {'gmm_tied_k10': 0.90071, 'kmeans_k8': 0.90071, 'dbscan_normalized_C100': 0.89362, 'ani_train_only': 0.91489, 'dbscan_train_only': 0.92199}


repeat6_fold4 {'gmm_tied_k10': 0.90071, 'kmeans_k8': 0.88652, 'dbscan_normalized_C100': 0.89362, 'ani_train_only': 0.88652, 'dbscan_train_only': 0.9078}


## Repeat 8 of 8: three candidates and two frozen references

In [12]:
run_repeat(7)




repeat7_fold0 {'gmm_tied_k10': 0.86525, 'kmeans_k8': 0.88652, 'dbscan_normalized_C100': 0.89362, 'ani_train_only': 0.89362, 'dbscan_train_only': 0.85816}


repeat7_fold1 {'gmm_tied_k10': 0.91135, 'kmeans_k8': 0.89362, 'dbscan_normalized_C100': 0.85106, 'ani_train_only': 0.90071, 'dbscan_train_only': 0.89362}


repeat7_fold2 {'gmm_tied_k10': 0.91489, 'kmeans_k8': 0.92199, 'dbscan_normalized_C100': 0.92908, 'ani_train_only': 0.95035, 'dbscan_train_only': 0.93617}


repeat7_fold3 {'gmm_tied_k10': 0.83333, 'kmeans_k8': 0.83688, 'dbscan_normalized_C100': 0.84397, 'ani_train_only': 0.87234, 'dbscan_train_only': 0.84752}


repeat7_fold4 {'gmm_tied_k10': 0.87234, 'kmeans_k8': 0.90071, 'dbscan_normalized_C100': 0.89362, 'ani_train_only': 0.92908, 'dbscan_train_only': 0.87943}


## Aggregate sample F1 and paired comparisons

In [13]:
assert len(FOLD_ROWS)==200
folds=pd.DataFrame(FOLD_ROWS);repeat_rows=[];subgroup_rows=[]
ALL_RECIPES=REFERENCES+NEW_RECIPES
for repeat in range(8):
    for recipe in ALL_RECIPES:
        frames=[pd.read_csv(OUT/'predictions'/f'repeat{repeat}_fold{fold}__{recipe}.csv',float_precision='round_trip') for fold in range(5)]
        frame=pd.concat(frames,ignore_index=True)
        assert len(frame)==235 and frame.ID.is_unique
        assert set(frame.ID)==set(raw.loc[known,'ID'])
        actual=frame[[f'actual_{r}' for r in REASONS]].to_numpy(dtype=int)
        probability=frame[[f'probability_{r}' for r in REASONS]].to_numpy();cutoff=.3 if recipe=='ani_train_only' else .5
        repeat_rows.append(dict(repeat=repeat,recipe=recipe,**score(actual,probability,cutoff)))
        for group in ['original_training','former_validation']:
            mask=frame.source_group.to_numpy()==group
            subgroup_rows.append(dict(repeat=repeat,recipe=recipe,source_group=group,
                                      **score(actual[mask],probability[mask],cutoff)))
repeats=pd.DataFrame(repeat_rows);repeats.to_csv(OUT/'repeat_scores.csv',index=False)
pd.DataFrame(subgroup_rows).to_csv(OUT/'subgroup_scores.csv',index=False)
base_comparison=pd.read_csv(BASE/'comparison.csv').set_index('variant')
for reference in REFERENCES:
    r=repeats[repeats.recipe==reference]
    np.testing.assert_allclose([r.sample_f1.mean(),r.macro_f1.mean()],
                              base_comparison.loc[reference,['cv_sample_f1','cv_macro_f1']].to_numpy(dtype=float),atol=1e-14)
summary=[];pair_rows=[]
for recipe in ALL_RECIPES:
    r=repeats[repeats.recipe==recipe]
    row=dict(recipe=recipe,cv_sample_f1=float(r.sample_f1.mean()),repeat_sd=float(r.sample_f1.std(ddof=1)),
             cv_macro_f1=float(r.macro_f1.mean()),**{f'cv_f1_{reason}':float(r[f'f1_{reason}'].mean()) for reason in REASONS})
    for reference in REFERENCES:
        f=folds[folds.recipe==recipe].set_index('context').sample_f1
        rf=folds[folds.recipe==reference].set_index('context').sample_f1
        delta=f-rf
        rr=repeats[repeats.recipe==reference].set_index('repeat').sample_f1
        dr=r.set_index('repeat').sample_f1-rr
        pair_rows.append(dict(recipe=recipe,reference=reference,mean_delta=float(dr.mean()),
                              fold_wins=int((delta>1e-12).sum()),fold_ties=int((abs(delta)<=1e-12).sum()),
                              fold_losses=int((delta< -1e-12).sum()),repeat_wins=int((dr>1e-12).sum()),
                              repeat_ties=int((abs(dr)<=1e-12).sum()),repeat_losses=int((dr< -1e-12).sum())))
        key='ani' if reference=='ani_train_only' else 'dbscan'
        row[f'delta_vs_{key}']=float(dr.mean())
    summary.append(row)
comparison=pd.DataFrame(summary).sort_values('cv_sample_f1',ascending=False,kind='stable')
comparison.to_csv(OUT/'comparison.csv',index=False)
paired=pd.DataFrame(pair_rows);paired.to_csv(OUT/'paired_comparison.csv',index=False)
display(comparison)
display(paired[paired.recipe.isin(NEW_RECIPES)])



,recipe,cv_sample_f1,repeat_sd,cv_macro_f1,cv_f1_Safety,cv_f1_Funding,cv_f1_Efficacy,cv_f1_Recruitment,cv_f1_Other,delta_vs_ani,delta_vs_dbscan
0,ani_train_only,0.908865,0.006631,0.822043,0.907372,0.760623,0.869526,0.972481,0.600213,0.000000,0.020550
3,kmeans_k8,0.895089,0.010270,0.807697,0.869877,0.746902,0.861350,0.965086,0.595272,-0.013777,0.006773
4,dbscan_normalized_C100,0.892624,0.008962,0.819820,0.888272,0.795219,0.854302,0.963976,0.597332,-0.016241,0.004309
1,dbscan_train_only,0.888316,0.003882,0.807876,0.868164,0.753986,0.853972,0.956423,0.606836,-0.020550,0.000000
2,gmm_tied_k10,0.887553,0.014661,0.795465,0.840483,0.744717,0.837053,0.960512,0.594558,-0.021312,-0.000762


,recipe,reference,mean_delta,fold_wins,fold_ties,fold_losses,repeat_wins,repeat_ties,repeat_losses
4,gmm_tied_k10,ani_train_only,-0.021312,11,3,26,1,0,7
5,gmm_tied_k10,dbscan_train_only,-0.000762,17,7,16,5,0,3
6,kmeans_k8,ani_train_only,-0.013777,10,4,26,0,0,8
7,kmeans_k8,dbscan_train_only,0.006773,22,10,8,7,0,1
8,dbscan_normalized_C100,ani_train_only,-0.016241,9,7,24,0,0,8
9,dbscan_normalized_C100,dbscan_train_only,0.004309,20,1,19,5,0,3


## Rescore saved predictions and independently refit classifiers

In [14]:
checks=[]
for row in folds.itertuples():
    frame=pd.read_csv(OUT/'predictions'/f'{row.context}__{row.recipe}.csv',float_precision='round_trip')
    probability=frame[[f'probability_{r}' for r in REASONS]].to_numpy()
    actual=frame[[f'actual_{r}' for r in REASONS]].to_numpy(dtype=int)
    cutoff=.3 if row.recipe=='ani_train_only' else .5
    rescored=score(actual,probability,cutoff)
    np.testing.assert_allclose([rescored['sample_f1'],rescored['macro_f1']],[row.sample_f1,row.macro_f1],atol=1e-14)
    np.testing.assert_array_equal(decide(probability,cutoff)[0],frame[[f'predicted_{r}' for r in REASONS]])
    if row.recipe in NEW_RECIPES:
        dataset=pd.read_csv(OUT/'datasets'/f'{row.context}__{row.recipe}.csv',float_precision='round_trip')
        assert set(frame.ID).isdisjoint(dataset.ID) and set(dataset.ID).isdisjoint(raw.iloc[test].ID)
        assert dataset.verified_fit.sum()==188
    checks.append(dict(context=row.context,recipe=row.recipe,passed=True))
pd.DataFrame(checks).to_csv(OUT/'disk_score_checks.csv',index=False)

replays=[]
for recipe in NEW_RECIPES:
    packet=joblib.load(OUT/'models'/f'repeat0_fold0__{recipe}.joblib')
    frame=pd.read_csv(OUT/'datasets'/f'repeat0_fold0__{recipe}.csv',float_precision='round_trip')
    rows=frame.global_row.to_numpy(dtype=int)
    design=packet['scaler'].transform(X[rows])
    if packet['normalizer'] is not None:design=packet['normalizer'].transform(design)
    replay={**packet,'estimators':[]}
    with threadpool_limits(limits=1):
        for j,reason in enumerate(REASONS):
            weights=frame[f'normalized_weight_{reason}'].to_numpy();included=weights>0
            estimator=clone(packet['estimators'][j])
            with warnings.catch_warnings():
                warnings.simplefilter('error',ConvergenceWarning)
                estimator.fit(design[included],frame[reason].to_numpy(dtype=int)[included],sample_weight=weights[included])
            replay['estimators'].append(estimator)
    held_frame=pd.read_csv(OUT/'predictions'/f'repeat0_fold0__{recipe}.csv')
    held=held_frame.global_row.to_numpy(dtype=int)
    np.testing.assert_allclose(predict(replay,X[held]),predict(packet,X[held]),rtol=1e-8,atol=1e-10)
    replays.append(dict(recipe=recipe,context='repeat0_fold0',passed=True))
pd.DataFrame(replays).to_csv(OUT/'independent_replays.csv',index=False)
for relative,digest in PROTECTED.items():assert hashlib.sha256((DATA/relative).read_bytes()).hexdigest()==digest
assert len(FIT_AUDIT)==600 and len(checks)==200 and len(replays)==3
assert all(r['held_fit_overlap']==r['test_fit_overlap']==0 for r in ROLE_ROWS)
print('Verified three native reproduction controls;120 saved new models;200 disk scores;three independent refits;all fold/data isolation and protected hashes.')



Verified three native reproduction controls;120 saved new models;200 disk scores;three independent refits;all fold/data isolation and protected hashes.


## Save the experiment record

In [15]:
settings=dict(protocol=PROTOCOL,new_fold_models=120,binary_fits=600,reference_fold_scores_reused=80,
              all_evaluated_fold_scores=200,repeat_scores=40,independent_refits=3,
              references_reproduced=True,active_reference_not_replaced=True,no_new_submission=True,
              versions=dict(python=platform.python_version(),sklearn=sklearn.__version__,numpy=np.__version__,pandas=pd.__version__))
(OUT/'settings.json').write_text(json.dumps(settings,indent=2)+'\n')
lines=['# Other leading recipes on the fixed training-only benchmark','',
       'Three fixed contenders: GMM tied k=10 + logistic C1000; K-means k=8 + logistic C1000; '
       'DBSCAN + standardized/unit-length embeddings + logistic C100. Compare with the unchanged '
       'training-only Ani and DBSCAN references on the exact same 40 saved splits. No parameter search.','',
       '## Results','','| Recipe | Mean sample F1 | Repeat SD | Mean macro F1 | Change vs Ani | Change vs DBSCAN |',
       '|---|---:|---:|---:|---:|---:|']
for row in comparison.itertuples():
    lines.append(f'| {row.recipe} | {row.cv_sample_f1:.6f} | {row.repeat_sd:.6f} | {row.cv_macro_f1:.6f} | '
                 f'{row.delta_vs_ani:+.6f} | {row.delta_vs_dbscan:+.6f} |')
lines+=['','## Paired results for the three added candidates','',
        '| Candidate | Reference | Mean change | Fold wins / ties / losses | Repeat wins / ties / losses |',
        '|---|---|---:|---|---|']
for row in paired[paired.recipe.isin(NEW_RECIPES)].itertuples():
    lines.append(f'| {row.recipe} | {row.reference} | {row.mean_delta:+.6f} | '
                 f'{row.fold_wins}/{row.fold_ties}/{row.fold_losses} | {row.repeat_wins}/{row.repeat_ties}/{row.repeat_losses} |')
lines+=['','## Fixed procedure and fitting access','',
        'Use all 235 known labels in eight repeats of five folds, with the saved KFold seeds100–107 and exact existing '
        'assignments. Each fit uses188 verified labels and1014 originally unlabeled training trials. '
        'Every held-out trial and every Kaggle test trial is excluded from all fitting, preprocessing, clustering, '
        'cluster-label mapping, confidence calculations, and augmentation. Classifier scaler fits the188 verified rows; '
        'cluster scaler fits the1202 training-pool rows. No extra PCA for GMM or K-means.','',
        'GMM uses ten tied-covariance components,n_init3,max_iter500,tol1e-3,reg_covar1e-3,seed42. '
        'K-means uses k8,n_init10,max_iter300,tol1e-4,seed42. Both use the unchanged partial-label mapping '
        'and balanced pseudo-label weight-cap rules, rebuilding labels from fitting labels inside each fold. '
        'Weights normalize to188 per reason. Logistic solvers remain LBFGS,tol1e-4,max_iter5000; '
        'C1000 for GMM/K-means and C100 for normalized DBSCAN.','',
        'The normalized candidate uses exactly the corresponding saved DBSCAN fold targets/weights, '
        'followed by verified-only StandardScaler and row L2 Normalizer. The normalizer does not learn from held-out inputs. '
        'Threshold0.5 and highest-probability fallback stay fixed for all new recipes; Ani retains its native0.3 cutoff.','',
        '## Interpretation and reuse','',
        'Primary F1 pools235 out-of-fold predictions per repeat and averages the8 repeat scores. '
        'Fold/repeat win counts and SD are descriptive; repeated folds overlap and are not independent final tests. '
        'Recipe choices were made before these scores, with no post-score C/k/cutoff search. '
        'The repeatedly used labeled pool and random folds may not reproduce later Kaggle trials. '
        'Historical validation scores serve only to verify implementation, not as additional winning criteria. '
        'Keep the original Ani reference frozen while recording any candidate that improves it.','',
        '## Reproduction and saved artifacts','',
        'Notebook:part1/leading_recipes_cv.ipynb. protocol.json/PROTOCOL.md record the exact source recipes and '
        'source-fold checksum references before fitting. datasets/ saves every new fold’s ID/target/raw/normalized-weight '
        'dataset; models/ saves all120 new classifier packets; clustering/ saves80 cluster models, responsibilities, '
        'support and reason rates. predictions/ includes120 new and80 reference held-out prediction files. '
        'fit_audit.csv, generated_labels.csv, role_audit.csv, repeat/subgroup scores and paired comparisons preserve details. '
        'Shared raw features and fold assignments remain at ../paired_recipe_cv/, checked by protocol hashes.','',
        'All three native157-label controls reproduce their original probabilities and (for clustering) targets/weights. '
        'Every new model reloads with matching probabilities. All200 saved prediction sets independently rescore. '
        'All three first-fold classifiers independently refit from persisted targets/weights. Original models, '
        'datasets, source folds and references retain their hashes. No submission CSV, upload, commit, push, or tag was created.']
(OUT/'SUMMARY.md').write_text('\n'.join(lines)+'\n')
print('Saved:',OUT/'SUMMARY.md')


Saved: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/experiments/leading_recipes_cv/SUMMARY.md


## Observed interpretation

Training-only Ani remains first at **0.908865** sample F1. The strongest newly tested alternative is K-means k8 at **0.895089**, followed by normalized DBSCAN C100 at **0.892624**. Original DBSCAN scores **0.888316**, and GMM tied k10 scores **0.887553**.

K-means improves on original DBSCAN by 0.006773, winning 22 folds, tying ten, and losing eight; its repeat mean is higher in seven of eight repeats. Normalization improves by 0.004309, winning 20 folds and losing 19, with one tie; its repeat mean is higher in five repeats. GMM is close on mean but lower by 0.000762 and has greater repeat variation.

None of the three added recipes beats Ani on mean sample F1. Ani wins all eight repeat averages against K-means and normalization, and seven of eight against this GMM. Keep Ani as the fixed reference for subsequent experiments. These results change the ranking relative to the old 78-trial comparison; they do not establish a new Kaggle ranking or guarantee transfer.
